## Setup: Creating sample documents

In [1]:
import os

os.makedirs("docs", exist_ok=True)

documents = {
    "doc1": "The cat sat on the mat and looked at the dog.",
    "doc2": "Dogs and cats are popular pets around the world.",
    "doc3": "Information retrieval systems help users find relevant documents.",
    "doc4": "A search engine uses an inverted index to retrieve documents quickly.",
    "doc5": "The dog chased the cat around the garden.",
    "doc6": "Boolean retrieval uses AND, OR, and NOT operators on query terms.",
    "doc7": "Machine learning models can improve search engine ranking.",
    "doc8": "Cats, dogs, and other pets require food, water, and care."
}

for doc_id, text in documents.items():
    with open(f"docs/{doc_id}.txt", "w") as f:
        f.write(text)

print(f"Created {len(documents)} documents in ./docs/")
for doc_id, text in documents.items():
    print(f"{doc_id}: {text}")

Created 8 documents in ./docs/
doc1: The cat sat on the mat and looked at the dog.
doc2: Dogs and cats are popular pets around the world.
doc3: Information retrieval systems help users find relevant documents.
doc4: A search engine uses an inverted index to retrieve documents quickly.
doc5: The dog chased the cat around the garden.
doc6: Boolean retrieval uses AND, OR, and NOT operators on query terms.
doc7: Machine learning models can improve search engine ranking.
doc8: Cats, dogs, and other pets require food, water, and care.


## Preprocessing (tokenize, lowercase, remove stopwords)

In [2]:
import re
import string

STOPWORDS = {
    "the", "a", "an", "and", "or", "not", "on", "at", "in", "of",
    "to", "is", "are", "can", "other", "around"
}

def preprocess(text):
    """Lowercase, strip punctuation, tokenize, remove stopwords."""
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = text.split()
    tokens = [t for t in tokens if t not in STOPWORDS]
    return tokens

# quick test
sample_tokens = preprocess(documents["doc1"])
print("doc1 tokens:", sample_tokens)

doc1 tokens: ['cat', 'sat', 'mat', 'looked', 'dog']


## Load and preprocess all documents

In [3]:
doc_tokens = {}  # doc_id -> list of tokens

for doc_id, text in documents.items():
    doc_tokens[doc_id] = preprocess(text)

for doc_id, tokens in doc_tokens.items():
    print(f"{doc_id}: {tokens}")

doc1: ['cat', 'sat', 'mat', 'looked', 'dog']
doc2: ['dogs', 'cats', 'popular', 'pets', 'world']
doc3: ['information', 'retrieval', 'systems', 'help', 'users', 'find', 'relevant', 'documents']
doc4: ['search', 'engine', 'uses', 'inverted', 'index', 'retrieve', 'documents', 'quickly']
doc5: ['dog', 'chased', 'cat', 'garden']
doc6: ['boolean', 'retrieval', 'uses', 'operators', 'query', 'terms']
doc7: ['machine', 'learning', 'models', 'improve', 'search', 'engine', 'ranking']
doc8: ['cats', 'dogs', 'pets', 'require', 'food', 'water', 'care']


## Build the dictionary (set of unique terms)

In [4]:
dictionary = set()

for tokens in doc_tokens.values():
    dictionary.update(tokens)

dictionary = sorted(dictionary)

print(f"Dictionary size: {len(dictionary)} unique terms")
print(dictionary)

Dictionary size: 40 unique terms
['boolean', 'care', 'cat', 'cats', 'chased', 'documents', 'dog', 'dogs', 'engine', 'find', 'food', 'garden', 'help', 'improve', 'index', 'information', 'inverted', 'learning', 'looked', 'machine', 'mat', 'models', 'operators', 'pets', 'popular', 'query', 'quickly', 'ranking', 'relevant', 'require', 'retrieval', 'retrieve', 'sat', 'search', 'systems', 'terms', 'users', 'uses', 'water', 'world']


## Build the inverted index

In [5]:
inverted_index = {term: set() for term in dictionary}

for doc_id, tokens in doc_tokens.items():
    for term in tokens:
        inverted_index[term].add(doc_id)

# pretty print
for term in sorted(inverted_index):
    postings = sorted(inverted_index[term])
    print(f"{term:15} -> {postings}")

boolean         -> ['doc6']
care            -> ['doc8']
cat             -> ['doc1', 'doc5']
cats            -> ['doc2', 'doc8']
chased          -> ['doc5']
documents       -> ['doc3', 'doc4']
dog             -> ['doc1', 'doc5']
dogs            -> ['doc2', 'doc8']
engine          -> ['doc4', 'doc7']
find            -> ['doc3']
food            -> ['doc8']
garden          -> ['doc5']
help            -> ['doc3']
improve         -> ['doc7']
index           -> ['doc4']
information     -> ['doc3']
inverted        -> ['doc4']
learning        -> ['doc7']
looked          -> ['doc1']
machine         -> ['doc7']
mat             -> ['doc1']
models          -> ['doc7']
operators       -> ['doc6']
pets            -> ['doc2', 'doc8']
popular         -> ['doc2']
query           -> ['doc6']
quickly         -> ['doc4']
ranking         -> ['doc7']
relevant        -> ['doc3']
require         -> ['doc8']
retrieval       -> ['doc3', 'doc6']
retrieve        -> ['doc4']
sat             -> ['doc1']
search      

## Boolean retrieval functions

In [6]:
ALL_DOCS = set(documents.keys())

def get_postings(term):
    term = term.lower()
    return inverted_index.get(term, set())

def bool_and(term1, term2):
    return get_postings(term1) & get_postings(term2)

def bool_or(term1, term2):
    return get_postings(term1) | get_postings(term2)

def bool_not(term):
    return ALL_DOCS - get_postings(term)

## Simple query parser (supports AND, OR, NOT, left to right)

In [7]:
def process_query(query):
    """
    Supports queries like:
    'cat'
    'cat AND dog'
    'cat OR fish'
    'NOT cat'
    'cat AND NOT dog'
    'dog OR cat AND NOT garden'   (evaluated strictly left to right)
    """
    tokens = query.strip().split()
    tokens = [t if t in ("AND", "OR", "NOT") else t.lower() for t in tokens]

    i = 0
    negate_next = False
    if tokens[0] == "NOT":
        negate_next = True
        i = 1

    result = get_postings(tokens[i])
    if negate_next:
        result = ALL_DOCS - result
    i += 1

    while i < len(tokens):
        op = tokens[i]
        i += 1
        negate_next = False
        if tokens[i] == "NOT":
            negate_next = True
            i += 1
        term_result = get_postings(tokens[i])
        if negate_next:
            term_result = ALL_DOCS - term_result
        i += 1

        if op == "AND":
            result = result & term_result
        elif op == "OR":
            result = result | term_result

    return result

## Run sample queries

In [8]:
test_queries = [
    "cat",
    "cat AND dog",
    "cat OR fish",
    "NOT cat",
    "cat AND NOT dog",
    "search AND engine",
    "pets OR machine"
]

for q in test_queries:
    result = process_query(q)
    print(f"Query: '{q}'")
    print(f"  Matching docs: {sorted(result) if result else 'None'}")
    for doc_id in sorted(result):
        print(f"    {doc_id}: {documents[doc_id]}")
    print()

Query: 'cat'
  Matching docs: ['doc1', 'doc5']
    doc1: The cat sat on the mat and looked at the dog.
    doc5: The dog chased the cat around the garden.

Query: 'cat AND dog'
  Matching docs: ['doc1', 'doc5']
    doc1: The cat sat on the mat and looked at the dog.
    doc5: The dog chased the cat around the garden.

Query: 'cat OR fish'
  Matching docs: ['doc1', 'doc5']
    doc1: The cat sat on the mat and looked at the dog.
    doc5: The dog chased the cat around the garden.

Query: 'NOT cat'
  Matching docs: ['doc2', 'doc3', 'doc4', 'doc6', 'doc7', 'doc8']
    doc2: Dogs and cats are popular pets around the world.
    doc3: Information retrieval systems help users find relevant documents.
    doc4: A search engine uses an inverted index to retrieve documents quickly.
    doc6: Boolean retrieval uses AND, OR, and NOT operators on query terms.
    doc7: Machine learning models can improve search engine ranking.
    doc8: Cats, dogs, and other pets require food, water, and care.

Quer